In [1]:
import pandas as pd
# Usamos latin-1 porque pueda tener acentos y la letra ñ
df = pd.read_csv('RNPDNO-22-08-2023.csv', encoding='latin-1')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 116945 entries, 0 to 116944
Data columns (total 11 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Consecutivo Reportes por Persona  110965 non-null  float64
 1   Consecutivo Registro              116945 non-null  int64  
 2   Nombre                            116939 non-null  str    
 3   Primer Apellido                   116906 non-null  str    
 4   Segundo Apellido                  114914 non-null  str    
 5   Edad                              108670 non-null  str    
 6   Sexo                              116945 non-null  str    
 7   Nacionalidad                      116943 non-null  str    
 8   Fecha de desaparición             57348 non-null   str    
 9   Entidad de desaparición           116945 non-null  str    
 10  Autoridad que reportó             116945 non-null  str    
dtypes: float64(1), int64(1), str(9)
memory usage: 9.8 MB


Podemos notar a primera instancia que al leer el .csv nos indica que tenemos un registro de 116945. El marco de la edad esta como tipo de texto en lugar de numero y podremos modificar el formato de fecha para facilitar un analisis
## 1. Checar duplicados
**Explicacion:** Veremos si hay duplicados cotejando su informacion dado que puede estar la misma persona registrada dos veces. Dado que si no lo hacemos el analisis sera erroneo



In [2]:
columna_cotejo=['Nombre', 'Primer Apellido', 'Segundo Apellido', 'Edad', 'Sexo', 'Fecha de desaparición']
print(f"Cantidad de duplicados: {df.duplicated(subset=columna_cotejo).sum()}")

Cantidad de duplicados: 38587


Eliminaremos los duplicados

In [3]:
df = df.drop_duplicates(subset=columna_cotejo,keep='first')

## 2. Coversion de Tipos
Ahora vamos a solucionar el problema que se tome la edad como cadena de texto, la vamos a transformar.

In [4]:
df['Edad'] = pd.to_numeric(df['Edad'],errors='coerce')
df['Edad'] = df['Edad'].astype('Int64')

Seguimos con la fecha de desaparicion donde modificaremos el formato, solo que hay tener en cuenta que nosotros usamos el formato mexicano poniendo primero el dia/mes/año

In [5]:
df['Fecha de desaparición'] = pd.to_datetime(df['Fecha de desaparición'], errors='coerce',dayfirst=True)

## 3. Normalizacion de Categorias
Necesitamos tener el mismo formato por lo cual hay que estandarizarlo

In [6]:
import numpy as np
columna_texto= ['Nombre', 'Primer Apellido', 'Segundo Apellido', 'Sexo', 
              'Nacionalidad', 'Entidad de desaparición', 'Autoridad que reportó']
for i in columna_texto:
    #Aqui converitmos a mayusculas y quitamos los espacios
    df[i]= df[i].astype(str).str.upper().str.strip()
    #Si encontramos valores nulos, regresamos 'NAN'
    df[i]=df[i].replace('NAN',np.nan)
    

## 4. Validaciones
Checamos si hay rangos razonables, fechas imposibles, coordenadas geográ-
ficas, en caso de que existan, u otras inconsistencias relevantes.
Primero busquemos fechas que son imposibles osea fechas despues del corte del dataset por lo que vemos que sea fechas despues del 22 de agosto del 2023.
De manera intuitiva tambien que sean edades realistas

In [7]:
import pandas as pd
import numpy as np
#Primero verificamos la fecha
#Le decimos a panda que usamos el formato en dd/mm/aaaa
df['Fecha de desaparición'] = pd.to_datetime(df['Fecha de desaparición'], dayfirst=False, errors='coerce')
#Establecemos nuestra fecha limite
fecha_corte = pd.to_datetime('2023-08-22')
#Filtramos
fechas_futuras = df[df['Fecha de desaparición'] > fecha_corte].shape[0]
print(f"Fechas futuras detectadas (reemplazadas por NaT): {fechas_futuras}")
#Reemplazamos las fechas imposibles por nulos
df.loc[df['Fecha de desaparición'] > fecha_corte, 'Fecha de desaparición'] = pd.NaT

#Verificamos las edades
edades_incoherentes = df[(df['Edad'] < 0) | (df['Edad'] > 120)].shape[0]
print(f"Edades fuera de rango (reemplazadas por NaN): {edades_incoherentes}")
df.loc[(df['Edad'] < 0) | (df['Edad'] > 120), 'Edad'] = np.nan

Fechas futuras detectadas (reemplazadas por NaT): 0
Edades fuera de rango (reemplazadas por NaN): 6


## Validaciones IMPORTANTES
Al analizar mas a profundidad, nos dimos cuenta que hay registros con el formato ELIMINADO1,ELIMINADO2,...,ELIMINADO7, nos planteamos que podria ser. Hay varios caminos, si unicamente tiene ELIMINADO en el campo del nombre junto con los apellidos significa que se tiene bajo el anonimato pero sigue contando como una desaparicion en cambio si tienen los campos relevantes(osea si tienen los datos de la persona como ELIMINADO) no podemos concluir algo acerca de ello por lo que generaria ruido.

Otras validaciones que se pudo checar es que puede darse el caso en el que el sexo no este en los parametros, por lo cual se tomara 
Tambien hay que ver si la entidad con la que se registro en verdad existe.

In [8]:
#Campos aceptados
entidades_mexico = [
    "AGUASCALIENTES", "BAJA CALIFORNIA", "BAJA CALIFORNIA SUR", "CAMPECHE", 
    "CHIAPAS", "CHIHUAHUA", "CIUDAD DE MEXICO", "COAHUILA", "COLIMA", 
    "DURANGO", "GUANAJUATO", "GUERRERO", "HIDALGO", "JALISCO", "MEXICO","ESTADO DE MEXICO", 
    "MICHOACAN", "MORELOS", "NAYARIT", "NUEVO LEON", "OAXACA", "PUEBLA", 
    "QUERETARO", "QUINTANA ROO", "SAN LUIS POTOSI", "SINALOA", "SONORA", 
    "TABASCO", "TAMAULIPAS", "TLAXCALA", "VERACRUZ", "YUCATAN", "ZACATECAS","SE DESCONOCE"
]
sexos_validos=[
    "HOMBRE","MUJER","INDETERMINADO","SE DESCONOCE"
]
#Tratamiento de registros "ELIMINADO"
#Primero, si columnas vitales como Sexo, Entidad dicen ELIMINADO, borramos la fila
condicion_insalvable = (
    df['Sexo'].str.contains('ELIMINADO', na=False) & 
    df['Entidad de desaparición'].str.contains('ELIMINADO', na=False) &
    df['Nacionalidad'].str.contains('ELIMINADO', na=False)
)

filas_insalvables = df[condicion_insalvable].shape[0]
print(f"Registros totalmente censurados (insalvables) borrados: {filas_insalvables}")

# Filtramos el dataset para quedarnos solo con los que NO cumplen esa condición
df = df[~condicion_insalvable]
#Segundo, si el registro se salvó pero el nombre dice ELIMINADO, lo cambiamos a ANONIMO
cols_nombres = ['Nombre', 'Primer Apellido', 'Segundo Apellido']
for col in cols_nombres:
    df.loc[df[col].str.contains('ELIMINADO', na=False), col] = 'ANONIMO'
#Verificacion si el sexo esta correcto
df.loc[~df['Sexo'].isin(sexos_validos), 'Sexo'] = 'SE DESCONOCE'
#Verificacion si las entidades federativas estan correctas
df.loc[~df['Entidad de desaparición'].isin(entidades_mexico), 'Entidad de desaparición'] = 'SE DESCONOCE'

Registros totalmente censurados (insalvables) borrados: 17


## 5. Tratamiento de Valores Faltantes
Vamos a ver cuantos son los datos de nulos de cada categoria para ver como tratarlos

In [9]:
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100
print("--- Porcentaje de valores faltantes por columna ---")
print(porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False))
print("-" * 50)

--- Porcentaje de valores faltantes por columna ---
Fecha de desaparición               70.429277
Edad                                 5.719866
Segundo Apellido                     1.840671
Consecutivo Reportes por Persona     1.395183
Primer Apellido                      0.033188
Nombre                               0.006382
Nacionalidad                         0.002553
dtype: float64
--------------------------------------------------


Analizando los porcentajes nos dimos cuenta que el mayor porcentaje se lo lleva la fecha de desaparicion por lo que aqui hay una decision muy importante. Se decidio dejar los `"valores vacios"` para dar a conocer que muchos casos de desaparicion no cuentan con fecha ademas que si las llegaramos a quitar, segaria demasiado nuetro analisis.

Para las variables de texto, se dejaran como `"SE DESCONOCE"` ayudando asi a mantener la integridad de los datos.

Las variables numericas se mantendran como `NaN` dado que si las usamos para los promedios o ceros afectara considerablemente a las medidas de tendencia central y de dispersion.

In [10]:
#Pasamos explícitamente 'object' y 'str' para evitar el warning de Pandas 3.0
columnas_texto = df.select_dtypes(include=['object', 'str']).columns

#Rellenamos los nulos de todas esas columnas con 'SE DESCONOCE'
df[columnas_texto] = df[columnas_texto].fillna('SE DESCONOCE')

print(f"Se han limpiado {len(columnas_texto)} columnas de texto.")

Se han limpiado 7 columnas de texto.


Pasamos el formato de la fecha al mexicano y verificamos la limpieza

In [11]:
if pd.api.types.is_datetime64_any_dtype(df['Fecha de desaparición']):
    df['Fecha de desaparición'] = df['Fecha de desaparición'].dt.strftime('%d/%m/%Y')

#Verificamos cómo quedaron los nulos depues de la limpieza
print("\n--- Valores faltantes DESPUÉS de la limpieza ---")
print(df.isnull().sum()[df.isnull().sum() > 0])


--- Valores faltantes DESPUÉS de la limpieza ---
Consecutivo Reportes por Persona     1093
Edad                                 4481
Fecha de desaparición               55175
dtype: int64


## 6. Exportamos

In [12]:
df.to_csv('RNPDNO_limpio.csv', index=False, encoding='utf-8')
print("\n¡Archivo 'RNPDNO_limpio.csv' generado exitosamente.")


¡Archivo 'RNPDNO_limpio.csv' generado exitosamente.
